In [ ]:
import pandas as pd
import spacy
from pysentimiento import create_analyzer
from spacy.matcher import PhraseMatcher

# Load spaCy model
nlp = spacy.load("en_core_web_md")

# Load pysentimiento analyzer - downloads model on first run, cached after
analyzer = create_analyzer(task="sentiment", lang="en")

# Matcher setup
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
terms = ["roofing", "siding", "skylight", "skylights", "gutter", "gutters", "vinyl", 
         "hardie", "ice dam", "flashing", "trim", "leak", "storm", "storms", "damage", 
         "cleanup", "dumpster", "estimate", "repair", "replace", "replacement", "warranty", 
         "crew", "insurance", "contractor", "inspection", "shingles", "project manager", 
         "permit", "permits", "ridge", "walkthrough", "punctual", "on time", "communication", 
         "quote", "assessment", "assessments", "claim", "claims", "downspout", "roof", 
         "door", "doors", "window", "windows"]
patterns = [nlp.make_doc(text) for text in terms]
matcher.add("CONTRACTOR_SERVICES", patterns)

# Load and combine sheets
sheets = pd.read_excel('roofing_sentiment_raw.xlsx', sheet_name=None) # Source data not included in repo to protect privacy

company_ids = {
    'A': 'A',
    'B': 'B',
    'C': 'C',
    'D': 'D'
}

dfs = []
for sheet_name, df in sheets.items():
    df['company_id'] = company_ids[sheet_name]
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

def process_row(text):
    if pd.isna(text) or str(text).strip() == '':
        return pd.Series({
            'polarity': None,
            'sentiment_label': None,
            'pos_score': None,
            'neu_score': None,
            'neg_score': None,
            'entities': '',
            'entity_count': 0
        })

    result = analyzer.predict(str(text))
    
    # Convert to -1 to 1 float to match existing output shape
    polarity = result.probas['POS'] - result.probas['NEG']

    # Entity extraction
    doc = nlp(str(text))
    ignore_types = {'ORG', 'DATE', 'TIME', 'CARDINAL', 'ORDINAL'} # these types of entities are not necessary for this analysis
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ not in ignore_types]

    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end]
        entities.append((span.text, "CONTRACTOR_SERVICES"))

    seen = set()
    unique_entities = []
    for e in entities:
        if e[0].lower() not in seen:
            seen.add(e[0].lower())
            unique_entities.append(e)

    return pd.Series({
        'polarity': round(polarity, 4),
        'sentiment_label': result.output,        # POS / NEU / NEG
        'pos_score': round(result.probas['POS'], 4),
        'neu_score': round(result.probas['NEU'], 4),
        'neg_score': round(result.probas['NEG'], 4),
        'entities': '; '.join([f"{e[0]}({e[1]})" for e in unique_entities]),
        'entity_count': len(unique_entities)
    })

result = combined_df['reviews'].apply(process_row)
combined_df[['polarity', 'sentiment_label', 'pos_score', 'neu_score', 
             'neg_score', 'entities', 'entity_count']] = result

combined_df.to_csv('roofing_absa_analyzed.csv', index=False)
print(combined_df[['reviews', 'polarity', 'sentiment_label', 'entities']].head())

# Summary stats
summary = combined_df.groupby('company_id').agg(
    review_count=('polarity', 'count'),
    mean_polarity=('polarity', 'mean'),
    median_polarity=('polarity', 'median'),
    mean_entity_count=('entity_count', 'mean')
).round(4)

print(summary.to_string())

spread = combined_df.groupby('company_id')['polarity'].agg(
    min_polarity='min',
    max_polarity='max',
    std_polarity='std'
).round(4)

print(spread.to_string())

In [ ]:
# CELL 2 - Aspect sentiment analysis
# Runs after Cell 1. Uses the same nlp, matcher, and combined_df already in memory.
# Produces a second CSV: roofing_aspect_sentiment.csv

aspect_buckets = {
    # Quality of work
    "roofing": "quality", "siding": "quality", "flashing": "quality",
    "shingles": "quality", "ridge": "quality", "leak": "quality",
    "vinyl": "quality", "hardie": "quality", "skylight": "quality",
    "skylights": "quality", "repair": "quality", "replace": "quality",
    "replacement": "quality",
    # Timeline / Scheduling
    "punctual": "timeline", "on time": "timeline",
    "project manager": "timeline", "permit": "timeline",
    "permits": "timeline", "walkthrough": "timeline",
    # Communication
    "communication": "communication", "estimate": "communication",
    "quote": "communication", "assessment": "communication",
    "assessments": "communication",
    # Cleanup
    "cleanup": "cleanup", "dumpster": "cleanup",
    # Insurance / Claims
    "insurance": "insurance", "claim": "insurance", "claims": "insurance",
    "inspection": "insurance", "damage": "insurance", "storm": "insurance",
    "storms": "insurance",
    # Crew / Professionalism
    "crew": "crew", "contractor": "crew",
    # Warranty
    "warranty": "warranty",
    # Scope - job type identification only, excluded from sentiment scoring
    "gutter": "scope", "gutters": "scope", "downspout": "scope",
    "door": "scope", "doors": "scope", "window": "scope",
    "windows": "scope", "roof": "scope", "trim": "scope", "ice dam": "scope",
}

def extract_aspect_sentiments(df):
    records = []

    for _, row in df.iterrows():
        text = row['reviews']
        contractor = row['company_id']

        if pd.isna(text) or str(text).strip() == '':
            continue

        doc = nlp(str(text))

        for sent in doc.sents:
            sent_doc = nlp(sent.text)
            matches = matcher(sent_doc)

            # Collect unique aspects per sentence - avoids double-counting
            # if a term appears twice in the same sentence
            seen_aspects = set()

            for match_id, start, end in matches:
                matched_term = sent_doc[start:end].text.lower()
                bucket = aspect_buckets.get(matched_term, "general")

                if bucket in ("scope", "general"):
                    continue
                if bucket in seen_aspects:
                    continue

                sent_result = analyzer.predict(sent.text)
                sentiment = sent_result.probas['POS'] - sent_result.probas['NEG']

                records.append({
                    "contractor": contractor,
                    "aspect": bucket,
                    "term": matched_term,
                    "sentiment": sentiment,
                    "sentence": sent.text.strip()
                })

    return pd.DataFrame(records)


# --- Generate the aspect DataFrame ---
aspect_df = extract_aspect_sentiments(combined_df)
aspect_df.to_csv('roofing_aspect_sentiment.csv', index=False)
print(f"Aspect records generated: {len(aspect_df)}")
print(aspect_df.head(10).to_string(index=False))


# --- Summary: avg sentiment and mention count by contractor + aspect ---
summary_aspect = aspect_df.groupby(["contractor", "aspect"]).agg(
    avg_sentiment=("sentiment", "mean"),
    std_sentiment=("sentiment", "std"),    # high std on near-zero avg = mixed signals
    mention_count=("sentiment", "count")
).round(3).reset_index()

print("\nAspect Summary by Contractor:")
print(summary_aspect.to_string(index=False))


# --- Heatmap ---
import matplotlib.pyplot as plt
import seaborn as sns

pivot = aspect_df.groupby(["contractor", "aspect"])["sentiment"].mean().unstack()

aspect_order = ["quality", "communication", "timeline", "crew", "cleanup", "warranty", "insurance"]
pivot = pivot.reindex(columns=[a for a in aspect_order if a in pivot.columns])

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title("Aspect Sentiment by Contractor", fontsize=14, pad=12)
ax.set_xlabel("Aspect", fontsize=11)
ax.set_ylabel("Contractor", fontsize=11)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("aspect_sentiment_heatmap.png", dpi=150)
plt.show()

In [ ]:
cleanup_sentences = aspect_df[aspect_df['aspect'] == 'cleanup'][['contractor', 'sentiment', 'sentence']]
print(cleanup_sentences.sort_values(['contractor', 'sentiment']).to_string(index=False))